# RxRx2 GPU continuation: 3-channel strict test and 2-channel training

This notebook tests whether raw six-channel Cell Painting images improve broad immune-function screening over released embeddings. The primary evaluation is **treatment-held-out**: a treatment never appears in more than one split. It is deliberately difficult and must not be replaced by a random-image split.

Run `6ch` first. Run `3ch` and `2ch` only if the 6-channel result is meaningful. Each completed epoch writes a resumable checkpoint to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import tarfile, random, json, os, time
import numpy as np, pandas as pd

# Change only if your Drive folder differs.
DRIVE_ROOT = Path('/content/drive/MyDrive/SUDAL/rxrx2')
TAR_PATH = DRIVE_ROOT / 'data/rxrx2_raw_subset.tar'
META_PATH = DRIVE_ROOT / 'data/metadata.csv'
WORK_ROOT = Path('/content/rxrx2_gpu')
OUT = DRIVE_ROOT / 'results_raw_image_gpu'
OUT.mkdir(parents=True, exist_ok=True)
assert TAR_PATH.exists(), TAR_PATH
assert META_PATH.exists(), META_PATH
print('tar:', TAR_PATH, 'metadata:', META_PATH, 'output:', OUT)

In [ ]:
# Colab local disk is much faster than reading thousands of PNGs from Drive.
# This only extracts when /content was reset. Do not rerun unnecessarily.
if not WORK_ROOT.exists():
    WORK_ROOT.mkdir(parents=True)
    with tarfile.open(TAR_PATH) as tar:
        tar.extractall(WORK_ROOT, filter='data')
image_candidates = [p for p in WORK_ROOT.rglob('images') if p.is_dir()]
assert image_candidates, f'Could not find images below {WORK_ROOT}'
IMAGE_ROOT = image_candidates[0]
print('IMAGE_ROOT =', IMAGE_ROOT)

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models
from PIL import Image
from sklearn.metrics import f1_score, balanced_accuracy_score, confusion_matrix
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type == 'cuda', 'This notebook is intentionally GPU-only. In Colab choose Runtime > Change runtime type > GPU.'
print('GPU:', torch.cuda.get_device_name(0))

GROUPS = {
 'type_I_II_interferons': ['IFN-alpha-A','IFNB-1','IFNE','IFN-omega','IFN-gamma'],
 'il1_tnf_superfamily': ['IL-1-alpha','IL-1-beta','TNF-alpha','TNF-beta','IL-18','IL-33','IL-36-gamma','TL-1A'],
 'chemokines_cxc_cc': ['ENA-78','GCP-2','GRO-alpha','GRO-B','GRO-gamma','IL-8','IP-10','I-TAC','CCL-1','CCL14a','CCL21','CCL28','CCL-4','MCP-1','MCP-2','MCP-3'],
 'angiogenic_vascular_regulatory': ['Angiopoietin-1','Angiopoietin-2','Angiopoietin-4','VEGF-165','VEGFC','VEGF-D','EG-VEGF','PLGF-4'],
}
SPLITS = {
 'type_I_II_interferons': {'train':['IFN-omega','IFN-gamma','IFNE'], 'val':['IFNB-1'], 'test':['IFN-alpha-A']},
 'il1_tnf_superfamily': {'train':['IL-1-alpha','TNF-alpha','TL-1A','IL-1-beta','IL-18'], 'val':['IL-33'], 'test':['TNF-beta','IL-36-gamma']},
 'chemokines_cxc_cc': {'train':['I-TAC','MCP-3','GRO-gamma','GRO-alpha','MCP-1','IL-8','CCL-1','GCP-2','CCL21','CCL-4'], 'val':['CCL28','CCL14a'], 'test':['IP-10','MCP-2','GRO-B','ENA-78']},
 'angiogenic_vascular_regulatory': {'train':['PLGF-4','Angiopoietin-2','VEGF-D','VEGFC','Angiopoietin-1'], 'val':['EG-VEGF'], 'test':['VEGF-165','Angiopoietin-4']},
}
CLASS_NAMES = list(GROUPS); CLASS_TO_ID = {x:i for i,x in enumerate(CLASS_NAMES)}
rows = [(t, label, split) for label,d in SPLITS.items() for split,ts in d.items() for t in ts]
split_df = pd.DataFrame(rows, columns=['treatment','label','split'])
split_df.to_csv(OUT/'treatment_heldout_split.csv', index=False)

In [ ]:
meta = pd.read_csv(META_PATH)
df = meta.merge(split_df, on='treatment', how='inner', validate='many_to_one').copy()
df['label_id'] = df.label.map(CLASS_TO_ID)
# Keep only selected sites whose complete channel files exist in the downloaded subset.
def base_path(r): return IMAGE_ROOT / r.experiment / f'Plate{r.plate}' / f'{r.well}_s{r.site}_w'
df['base'] = [str(base_path(r)) for r in df.itertuples()]
df = df[[Path(b + '1.png').exists() and Path(b + '6.png').exists() for b in df.base]].reset_index(drop=True)
assert len(df) > 0, 'No matching images: check IMAGE_ROOT and metadata.'
print(df.groupby(['split','label']).size().unstack(fill_value=0))
print('sites retained:', len(df), 'treatments:', df.treatment.nunique())

In [ ]:
IMAGE_SIZE, BATCH_SIZE, EPOCHS, PATIENCE, NUM_WORKERS = 224, 32, 25, 5, 2
# Pre-registered channel ablation; not selected after looking at test performance.
# This continuation notebook evaluates the existing 3-channel checkpoint, then trains only 2ch.
RUN_CONFIGS = {'2ch_dna_mito': (1,5)}

class RxRx2Dataset(Dataset):
    def __init__(self, frame, channels, augment=False):
        self.rows = frame[['base','label_id','site_id']].to_dict('records'); self.channels = channels; self.augment = augment
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        xs=[]
        for c in self.channels:
            x=np.asarray(Image.open(r['base'] + f'{c}.png'), dtype=np.float32)/255.0
            xs.append(torch.from_numpy(x)[None])
        x=torch.cat(xs); x=torch.nn.functional.interpolate(x[None], size=(IMAGE_SIZE,IMAGE_SIZE), mode='bilinear', align_corners=False)[0]
        if self.augment:
            if torch.rand(()) < .5: x=x.flip(2)
            if torch.rand(()) < .5: x=x.flip(1)
            x=torch.rot90(x, int(torch.randint(0,4,()).item()), (1,2))
        return x, r['label_id'], r['site_id']

def loaders(channels):
    tr,va,te=(df[df.split==s].copy() for s in ['train','val','test'])
    counts=tr.label_id.value_counts(); weights=tr.label_id.map(lambda x: 1/counts[x]).to_numpy()
    sampler=WeightedRandomSampler(weights, len(weights), replacement=True)
    kw=dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=False)
    return (DataLoader(RxRx2Dataset(tr,channels,True), sampler=sampler, **kw),
            DataLoader(RxRx2Dataset(va,channels,False), shuffle=False, **kw),
            DataLoader(RxRx2Dataset(te,channels,False), shuffle=False, **kw))

class PretrainedMobileNet(nn.Module):
    def __init__(self, in_channels):
        super().__init__(); self.net=models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        old=self.net.features[0][0]; new=nn.Conv2d(in_channels,old.out_channels,old.kernel_size,old.stride,old.padding,bias=False)
        with torch.no_grad(): new.weight.copy_(old.weight.mean(1,keepdim=True).repeat(1,in_channels,1,1)*(3/in_channels))
        self.net.features[0][0]=new; self.net.classifier[3]=nn.Linear(self.net.classifier[3].in_features,4)
    def forward(self,x): return self.net(x)

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); y=[]; p=[]
    for x,yy,_ in loader:
        p.extend(model(x.to(DEVICE,non_blocking=True)).argmax(1).cpu().numpy()); y.extend(yy.numpy())
    return {'macro_f1':float(f1_score(y,p,average='macro')),'balanced_accuracy':float(balanced_accuracy_score(y,p)),'y':np.array(y),'p':np.array(p)}

In [ ]:
def run_config(name, channels):
    tag=f'pretrained_mobilenet_{name}_treatment_heldout'; best_path=OUT/f'{tag}_best.pt'; resume_path=OUT/f'{tag}_resume.pt'
    tr,va,te=loaders(channels); model=PretrainedMobileNet(len(channels)).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=1e-4,weight_decay=1e-4); scaler=torch.amp.GradScaler('cuda')
    start,best,wait,hist=1,-1.,0,[]
    if resume_path.exists():
        s=torch.load(resume_path,map_location=DEVICE); model.load_state_dict(s['model']); opt.load_state_dict(s['opt']); scaler.load_state_dict(s['scaler'])
        start,best,wait,hist=s['epoch']+1,s['best'],s['wait'],s['history']; print('resume at',start)
    for epoch in range(start,EPOCHS+1):
        model.train(); loss_sum=0.; n=0
        for x,y,_ in tqdm(tr,desc=f'{name} epoch {epoch}'):
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'): loss=nn.functional.cross_entropy(model(x.to(DEVICE,non_blocking=True)),y.to(DEVICE,non_blocking=True))
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); loss_sum+=loss.item(); n+=1
        val=evaluate(model,va); row={'epoch':epoch,'train_loss':loss_sum/n,**{f'val_{k}':v for k,v in val.items() if k not in ['y','p']}}; hist.append(row); print(row)
        if val['macro_f1']>best: best,wait=val['macro_f1'],0; torch.save({'model':model.state_dict(),'channels':channels},best_path)
        else: wait+=1
        torch.save({'epoch':epoch,'model':model.state_dict(),'opt':opt.state_dict(),'scaler':scaler.state_dict(),'best':best,'wait':wait,'history':hist},resume_path)
        pd.DataFrame(hist).to_csv(OUT/f'{tag}_history.csv',index=False)
        if wait>=PATIENCE: break
    model.load_state_dict(torch.load(best_path,map_location=DEVICE)['model']); test=evaluate(model,te)
    result={'model':'ImageNet-pretrained MobileNetV3-Small','setting':name,'channels':len(channels),'channel_ids':str(channels),'protocol':'both HUVEC, treatment-held-out','macro_f1':test['macro_f1'],'balanced_accuracy':test['balanced_accuracy']}
    pd.DataFrame([result]).to_csv(OUT/f'{tag}_summary.csv',index=False)
    pd.DataFrame(confusion_matrix(test['y'],test['p'],labels=range(4),normalize='true'),index=CLASS_NAMES,columns=CLASS_NAMES).to_csv(OUT/f'{tag}_confusion.csv')
    print(result); return result

# Evaluate the saved 3-channel model without further training.
def evaluate_saved_checkpoint(name, channels):
    tag = f'pretrained_mobilenet_{name}_treatment_heldout'
    best_path = OUT / f'{tag}_best.pt'
    assert best_path.exists(), f'Missing checkpoint: {best_path}. Copy/share the earlier results_raw_image_gpu folder into this Drive first.'
    _, _, test_loader = loaders(channels)
    model = PretrainedMobileNet(len(channels)).to(DEVICE)
    model.load_state_dict(torch.load(best_path, map_location=DEVICE)['model'])
    test = evaluate(model, test_loader)
    result = {'model':'ImageNet-pretrained MobileNetV3-Small', 'setting':name, 'channels':len(channels), 'channel_ids':str(channels), 'protocol':'both HUVEC, treatment-held-out', 'macro_f1':test['macro_f1'], 'balanced_accuracy':test['balanced_accuracy']}
    pd.DataFrame([result]).to_csv(OUT / f'{tag}_strict_test.csv', index=False)
    pd.DataFrame(confusion_matrix(test['y'], test['p'], labels=range(4), normalize='true'), index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(OUT / f'{tag}_strict_test_confusion.csv')
    print('3-channel strict test:', result)
    return result

result_3ch = evaluate_saved_checkpoint('3ch_dna_rna_mito', (1,4,5))
result_2ch = run_config('2ch_dna_mito', (1,5))
result_6ch = {'model':'ImageNet-pretrained MobileNetV3-Small', 'setting':'6ch', 'channels':6, 'channel_ids':'(1, 2, 3, 4, 5, 6)', 'protocol':'both HUVEC, treatment-held-out', 'macro_f1':0.402187554202103, 'balanced_accuracy':0.39985620471014494}
channel_summary = pd.DataFrame([result_6ch, result_3ch, result_2ch]).sort_values('channels', ascending=False)
channel_summary.to_csv(OUT / 'pretrained_channel_ablation_final_summary.csv', index=False)
channel_summary

## Interpretation rule

Compare raw-image CNN against the embedding baseline only under the same treatment-held-out protocol. Do not select a channel set from test performance; use validation for choices. If 6-channel CNN does not improve materially over the embedding baseline, report that raw-image fine-tuning did not overcome treatment/domain shift in this subset. Run the Transformer only after this baseline is complete and only as a secondary architecture comparison.